# Assignment Case A — HR Attrition & Workforce Analytics
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *(isi nama kamu)* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case A — HR Attrition & Workforce Analytics |
| **Source** | PostgreSQL AdventureWorks — HumanResources schema |
| **Target** | Hive: `adventureworks_curated.fact_hr_workforce` |

---

## ⚠️ Filter Penting

> **Kamu HANYA boleh menggunakan data dari Department: `Production` dan `Engineering`.**  
> Filter ini harus diterapkan sebelum transformasi.

## 🎯 Business Questions

1. **Department mana** (Production vs Engineering) yang memiliki rata-rata tenure karyawan lebih tinggi?
2. **Apakah ada korelasi** antara pay rate dan lama kerja karyawan?
3. **Berapa % karyawan** yang pernah pindah departemen minimal 1 kali?

## 📐 Pipeline yang Harus Dibangun

```
PostgreSQL (JDBC)  →  Spark DataFrame  →  Transform  →  Hive Curated Table
HumanResources.*      filter dept          TenureYears    fact_hr_workforce
                      join 5 tabel         IsMover
                                           DeptChangeCount
```

## 📋 Kolom Target: `fact_hr_workforce`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `employee_id` | INT | Primary key |
| `full_name` | STRING | FirstName + LastName |
| `job_title` | STRING | Jabatan karyawan |
| `department_name` | STRING | Nama departemen (Production / Engineering) |
| `hire_date` | DATE | Tanggal mulai kerja |
| `tenure_years` | DECIMAL(5,2) | Lama kerja dalam tahun dari HireDate s/d sekarang |
| `latest_pay_rate` | DECIMAL(10,2) | Pay rate terbaru dari EmployeePayHistory |
| `is_mover` | BOOLEAN | TRUE jika pernah pindah departemen >= 1x |
| `dept_change_count` | INT | Total berapa kali pindah departemen |
| `load_timestamp` | TIMESTAMP | Waktu load |

## 🔗 Source Tables

| Tabel | Schema | Keterangan |
|-------|--------|------------|
| Employee | HumanResources | Data karyawan: JobTitle, HireDate |
| EmployeeDepartmentHistory | HumanResources | Riwayat perpindahan dept (EndDate NULL = aktif) |
| EmployeePayHistory | HumanResources | Riwayat pay rate — ambil terbaru |
| Department | HumanResources | Nama departemen — filter Production & Engineering |
| Person | Person | Nama lengkap karyawan |


## 0. Persiapan: Buat Hive External Tables untuk HR

Sebelum mulai analisis, kamu perlu menyiapkan tabel sumber sebagai
**Hive External Tables**. Pola tutorialnya sama dengan:
- `02_adventureworks_to_hdfs.ipynb` → extract PostgreSQL ke HDFS Parquet
- `03_hdfs_to_hive.ipynb` → buat External Tables di atas Parquet

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL |
|---|---|
| `dim_employee` | `HumanResources.Employee` |
| `dim_emp_dept_history` | `HumanResources.EmployeeDepartmentHistory` |
| `dim_emp_pay_history` | `HumanResources.EmployeePayHistory` |
| `dim_department` | `HumanResources.Department` |
| `dim_person` | `Person.Person` |

> **Lewati bagian ini** jika tabel sudah ada di Hive.

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

Mirip dengan `02_adventureworks_to_hdfs.ipynb`. SparkSession ini
**tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet.

In [26]:
# TODO: setup environment & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# TODO: buat spark_jdbc tanpa Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('HR-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# TODO: konfigurasi JDBC PostgreSQL AdventureWorks
JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {
    'user': 'postgres',
    'password': 'My_password1',
    'driver': 'org.postgresql.Driver'
}
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'spark_jdbc {spark_jdbc.version} ready')


spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel HR ke HDFS sebagai Parquet

Mirip dengan section 2 di `02_adventureworks_to_hdfs.ipynb`.

In [27]:
# TODO: lengkapi list tabel HR yang harus di-extract
hr_tables = [
    ('HumanResources', 'Employee'),
    ('HumanResources', 'EmployeeDepartmentHistory'),
    ('HumanResources', 'EmployeePayHistory'),
    ('HumanResources', 'Department'),
    ('Person', 'Person'),
]

# TODO: loop, baca via JDBC, simpan ke HDFS sebagai Parquet
for schema, table in hr_tables:
    df = read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:30s} -> {df.count():6,} rows -> HDFS')

print('Semua tabel HR berhasil disimpan ke HDFS!')


  HumanResources.Employee                       ->    290 rows -> HDFS
  HumanResources.EmployeeDepartmentHistory      ->    296 rows -> HDFS
  HumanResources.EmployeePayHistory             ->    316 rows -> HDFS
  HumanResources.Department                     ->     16 rows -> HDFS
  Person.Person                         -> 19,972 rows -> HDFS
Semua tabel HR berhasil disimpan ke HDFS!


### 0.3 Setup SparkSession dengan Hive Support

Mirip dengan `03_hdfs_to_hive.ipynb`. Tutup `spark_jdbc` dulu, lalu buat
SparkSession baru **dengan** Hive support untuk membuat External Tables.

In [28]:
# TODO: stop spark_jdbc agar tidak konflik resource
spark_jdbc.stop()

from pyspark.sql import SparkSession

# TODO: buat spark_hive dengan Hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('HR-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# TODO: buat database adventureworks jika belum ada
spark_hive.sql('CREATE DATABASE IF NOT EXISTS adventureworks')
spark_hive.sql('USE adventureworks')
print(f'spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()


spark_hive 3.5.0 ready
+--------------------+
|           namespace|
+--------------------+
|      adventureworks|
|adventureworks_cu...|
|             default|
+--------------------+



### 0.4 Buat Hive External Tables

Mirip section 3 di `03_hdfs_to_hive.ipynb`. Schema otomatis di-infer dari Parquet.

In [29]:
# Helper function — sama persis dengan pola di 03_hdfs_to_hive.ipynb
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

# TODO: lengkapi mapping nama tabel Hive -> path HDFS
hive_tables = [
    ('dim_employee',         f'{HDFS_BASE}/HumanResources/Employee'),
    ('dim_emp_dept_history', f'{HDFS_BASE}/HumanResources/EmployeeDepartmentHistory'),
    ('dim_emp_pay_history',  f'{HDFS_BASE}/HumanResources/EmployeePayHistory'),
    ('dim_department',       f'{HDFS_BASE}/HumanResources/Department'),
    ('dim_person',           f'{HDFS_BASE}/Person/Person'),
]

# TODO: loop, panggil create_external_table
for tname, path in hive_tables:
    create_external_table(tname, path)

print()
spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('Setup selesai. Sekarang lanjut ke ## 0. Setup SparkSession utama.')


  dim_employee                        ->    290 rows
  dim_emp_dept_history                ->    296 rows
  dim_emp_pay_history                 ->    316 rows
  dim_department                      ->     16 rows
  dim_person                          -> 19,972 rows

+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|dim_emp_dept_history|      false|
|adventureworks|        dim_employee|      false|
|adventureworks| dim_emp_pay_history|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|  fact_order_details|      false|
|adventureworks|   fact_sales_orde

## 0. Setup SparkSession

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import date

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseA_HRAttrition') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .enableHiveSupport() \
    .getOrCreate()

# Referensi tanggal hari ini (untuk hitung TenureYears)
REFERENCE_DATE = date.today().isoformat()

# Filter department yang diperbolehkan
VALID_DEPARTMENTS = ['Production', 'Engineering']

print(f'Spark {spark.version} ready')
print(f'Reference date: {REFERENCE_DATE}')
print(f'Filter departments: {VALID_DEPARTMENTS}')
print('Spark UI → http://localhost:4040')

Spark 3.5.0 ready
Reference date: 2026-05-17
Filter departments: ['Production', 'Engineering']
Spark UI → http://localhost:4040


## 1. Load Data dari Hive External Tables

Setelah Bagian 0 selesai, semua tabel sumber sudah ada di Hive sebagai External Tables.
Baca dengan `spark.table('adventureworks.<nama_tabel>')`.

In [31]:
# Sumber data: Hive External Tables (sudah dibuat di Bagian 0 di atas)
# TODO: Load semua tabel dari Hive menggunakan spark.table()
df_employee   = spark.table('adventureworks.dim_employee')
df_edh        = spark.table('adventureworks.dim_emp_dept_history')
df_pay        = spark.table('adventureworks.dim_emp_pay_history')
df_dept       = spark.table('adventureworks.dim_department')
df_person     = spark.table('adventureworks.dim_person')

print('Row counts:')
for name, df in [
    ('dim_employee',         df_employee),
    ('dim_emp_dept_history', df_edh),
    ('dim_emp_pay_history',  df_pay),
    ('dim_department',       df_dept),
    ('dim_person',           df_person),
]:
    print(f'  {name}: {df.count():,}')


Row counts:
  dim_employee: 290
  dim_emp_dept_history: 296
  dim_emp_pay_history: 316
  dim_department: 16
  dim_person: 19,972


## 2. Eksplorasi Data

In [32]:
# TODO: Eksplorasi schema dan data
print('=== Schema Employee ===')
df_employee.printSchema()

print('\n=== Schema EmployeeDepartmentHistory ===')
df_edh.printSchema()

print('\n=== Semua Department yang tersedia ===')
df_dept.select('DepartmentID', 'Name', 'GroupName').orderBy('Name').show(truncate=False)

print('\n=== Distribusi karyawan per department ===')
df_edh.filter(F.col('EndDate').isNull()) \
    .join(df_dept, on='DepartmentID', how='inner') \
    .groupBy('Name') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(truncate=False)

=== Schema Employee ===
root
 |-- BusinessEntityID: integer (nullable = true)
 |-- NationalIDNumber: string (nullable = true)
 |-- LoginID: string (nullable = true)
 |-- OrganizationNode: string (nullable = true)
 |-- OrganizationLevel: integer (nullable = true)
 |-- JobTitle: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HireDate: date (nullable = true)
 |-- SalariedFlag: boolean (nullable = true)
 |-- VacationHours: short (nullable = true)
 |-- SickLeaveHours: short (nullable = true)
 |-- CurrentFlag: boolean (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)


=== Schema EmployeeDepartmentHistory ===
root
 |-- BusinessEntityID: integer (nullable = true)
 |-- DepartmentID: short (nullable = true)
 |-- ShiftID: short (nullable = true)
 |-- StartDate: date (nullable = true)
 |-- EndDate: date (nullable = true)
 |-- ModifiedDate: 

## 3. Extract — Filter & Join

**⚠️ Filter Department Production & Engineering harus diterapkan di sini!**

Pipeline join yang diharapkan:
```
Employee
    JOIN EmployeeDepartmentHistory (EndDate IS NULL = departemen aktif)
    JOIN Department (filter Name IN ['Production','Engineering'])
    JOIN Person (ambil FirstName, LastName)
    JOIN LatestPayRate (subquery: ROW_NUMBER() OVER PARTITION BY EmployeeID ORDER BY RateChangeDate DESC)
    JOIN DeptHistoryCount (subquery: COUNT(*) GROUP BY EmployeeID)
```

In [33]:
# ── Step 1: Filter department yang valid ─────────────────────────────────
df_dept_filtered = df_dept.filter(F.col('Name').isin(VALID_DEPARTMENTS))
print(f'Department yang lolos filter: {df_dept_filtered.count()}')
df_dept_filtered.show()

Department yang lolos filter: 2
+------------+-----------+--------------------+-------------------+
|DepartmentID|       Name|           GroupName|       ModifiedDate|
+------------+-----------+--------------------+-------------------+
|           1|Engineering|Research and Deve...|2008-04-30 00:00:00|
|           7| Production|       Manufacturing|2008-04-30 00:00:00|
+------------+-----------+--------------------+-------------------+



In [34]:
# ── Step 2: Ambil departemen aktif per karyawan (EndDate IS NULL) ─────────
# TODO: Filter df_edh dimana EndDate IS NULL
df_edh_active = df_edh.filter(F.col('EndDate').isNull())
print(f'Active dept history rows: {df_edh_active.count():,}')

Active dept history rows: 290


In [35]:
# ── Step 3: Hitung pay rate terbaru per karyawan ──────────────────────────
# Gunakan Window function: ROW_NUMBER() OVER (PARTITION BY BusinessEntityID ORDER BY RateChangeDate DESC)
# Ambil hanya row dengan rn = 1

win_pay = Window.partitionBy('BusinessEntityID').orderBy(F.desc('RateChangeDate'))

df_latest_pay = df_pay \
    .withColumn('rn', F.row_number().over(win_pay)) \
    .filter(F.col('rn') == 1) \
    .select(
        F.col('BusinessEntityID').alias('pay_emp_id'),
        F.col('Rate').alias('latest_pay_rate'),
        F.col('RateChangeDate').alias('rate_change_date')
    )

print(f'Latest pay rate rows: {df_latest_pay.count():,}')
df_latest_pay.show(5)

Latest pay rate rows: 290
+----------+---------------+-------------------+
|pay_emp_id|latest_pay_rate|   rate_change_date|
+----------+---------------+-------------------+
|         1|       125.5000|2009-01-14 00:00:00|
|         2|        63.4615|2008-01-31 00:00:00|
|         3|        43.2692|2007-11-11 00:00:00|
|         4|        29.8462|2011-12-15 00:00:00|
|         5|        32.6923|2008-01-06 00:00:00|
+----------+---------------+-------------------+
only showing top 5 rows



In [36]:
# ── Step 4: Hitung total dept history count per karyawan ─────────────────
# DeptHistoryCount = COUNT(*) GROUP BY BusinessEntityID dari tabel EmployeeDepartmentHistory
# (semua history, bukan hanya yang aktif)

# TODO: groupBy BusinessEntityID, hitung count semua record
df_hist_count = df_edh \
    .groupBy('BusinessEntityID') \
    .agg(F.count('*').alias('dept_history_count'))

print(f'History count rows: {df_hist_count.count():,}')
df_hist_count.show(5)

History count rows: 290
+----------------+------------------+
|BusinessEntityID|dept_history_count|
+----------------+------------------+
|             148|                 1|
|             243|                 1|
|              31|                 1|
|              85|                 1|
|             137|                 1|
+----------------+------------------+
only showing top 5 rows



In [37]:
# ── Step 5: Join semua tabel ──────────────────────────────────────────────
# TODO: Gabungkan semua tabel
# Join order:
#   1. df_employee JOIN df_edh_active ON BusinessEntityID
#   2. JOIN df_dept_filtered ON DepartmentID
#   3. JOIN df_person ON BusinessEntityID
#   4. JOIN df_latest_pay ON BusinessEntityID
#   5. JOIN df_hist_count ON BusinessEntityID

df_enriched = df_employee \
    .join(
        df_edh_active.select('BusinessEntityID', 'DepartmentID'),
        on='BusinessEntityID',
        how='inner'
    ) \
    .join(
        df_dept_filtered.select(F.col('DepartmentID'), F.col('Name').alias('dept_name')),
        on='DepartmentID',
        how='inner'
    ) \
    .join(
        df_person.select(
            F.col('BusinessEntityID').alias('person_id'),
            'FirstName',
            'LastName'
        ),
        df_employee['BusinessEntityID'] == F.col('person_id'),
        how='left'
    ) \
    .join(
        df_latest_pay,
        df_employee['BusinessEntityID'] == df_latest_pay['pay_emp_id'],
        how='left'
    ) \
    .join(
        df_hist_count,
        on='BusinessEntityID',
        how='left'
    )

print(f'Total rows setelah join: {df_enriched.count():,}')
df_enriched.printSchema()
df_enriched.show(5, truncate=False)

Total rows setelah join: 185
root
 |-- BusinessEntityID: integer (nullable = true)
 |-- DepartmentID: short (nullable = true)
 |-- NationalIDNumber: string (nullable = true)
 |-- LoginID: string (nullable = true)
 |-- OrganizationNode: string (nullable = true)
 |-- OrganizationLevel: integer (nullable = true)
 |-- JobTitle: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HireDate: date (nullable = true)
 |-- SalariedFlag: boolean (nullable = true)
 |-- VacationHours: short (nullable = true)
 |-- SickLeaveHours: short (nullable = true)
 |-- CurrentFlag: boolean (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)
 |-- dept_name: string (nullable = true)
 |-- person_id: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- pay_emp_id: integer (nullable = true)
 |-- latest_pay_ra

## 4. Transform — Hitung Kolom Turunan

### Aturan Kalkulasi:
- **tenure_years** = `ROUND((DATE_PART('year', AGE(NOW(), HireDate)) + DATE_PART('month', AGE(NOW(), HireDate))/12.0 + DATE_PART('day', AGE(NOW(), HireDate))/365.0), 2)`
- **is_mover** = `TRUE` jika `dept_history_count > 1`
- **dept_change_count** = `GREATEST(dept_history_count - 1, 0)` (posisi awal tidak dihitung)
- **full_name** = `CONCAT(FirstName, ' ', LastName)`

In [38]:
# TODO: Hitung semua kolom turunan
# Petunjuk untuk tenure_years:
# F.datediff(F.current_date(), F.col('HireDate')) / 365.25
# Atau lebih presisi:
# (F.months_between(F.current_date(), F.col('HireDate')) / 12)

df_transformed = df_enriched \
    .withColumn('full_name',
        F.concat_ws(' ', F.col('FirstName'), F.col('LastName'))) \
    .withColumn('tenure_years',
        F.round(F.months_between(F.current_date(), F.col('HireDate')) / 12, 2)) \
    .withColumn('latest_pay_rate',
        F.round(F.col('latest_pay_rate'), 2)) \
    .withColumn('is_mover',
        F.col('dept_history_count') > 1) \
    .withColumn('dept_change_count',
        F.greatest(F.col('dept_history_count') - 1, F.lit(0)).cast('int')) \
    .select(
        F.col('BusinessEntityID').alias('employee_id'),
        F.col('full_name'),
        F.col('JobTitle').alias('job_title'),
        F.col('dept_name').alias('department_name'),
        F.col('HireDate').alias('hire_date'),
        F.col('tenure_years'),
        F.col('latest_pay_rate'),
        F.col('is_mover'),
        F.col('dept_change_count')
    )

print(f'Total rows setelah transform: {df_transformed.count():,}')
df_transformed.printSchema()
df_transformed.show(10, truncate=False)

Total rows setelah transform: 185
root
 |-- employee_id: integer (nullable = true)
 |-- full_name: string (nullable = false)
 |-- job_title: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- tenure_years: double (nullable = true)
 |-- latest_pay_rate: decimal(18,2) (nullable = true)
 |-- is_mover: boolean (nullable = true)
 |-- dept_change_count: integer (nullable = false)

+-----------+------------------+-----------------------------+---------------+----------+------------+---------------+--------+-----------------+
|employee_id|full_name         |job_title                    |department_name|hire_date |tenure_years|latest_pay_rate|is_mover|dept_change_count|
+-----------+------------------+-----------------------------+---------------+----------+------------+---------------+--------+-----------------+
|2          |Terri Duffy       |Vice President of Engineering|Engineering    |2008-01-31|18.3        |63.46          |f

In [39]:
# Verifikasi filter department sudah benar
print('=== Department yang ada di hasil ===')
df_transformed.groupBy('department_name').count().show()

# Verifikasi IsMover
print('\n=== Distribusi IsMover ===')
df_transformed.groupBy('is_mover').count().show()

# Verifikasi tenure_years range
print('\n=== TenureYears Statistics ===')
df_transformed.select('tenure_years').describe().show()

=== Department yang ada di hasil ===
+---------------+-----+
|department_name|count|
+---------------+-----+
|    Engineering|    6|
|     Production|  179|
+---------------+-----+


=== Distribusi IsMover ===
+--------+-----+
|is_mover|count|
+--------+-----+
|   false|  185|
+--------+-----+


=== TenureYears Statistics ===
+-------+------------------+
|summary|      tenure_years|
+-------+------------------+
|  count|               185|
|   mean|17.224648648648646|
| stddev|0.5619618302940187|
|    min|             15.33|
|    max|             19.88|
+-------+------------------+



## 5. Window Functions — Ranking per Department

In [40]:
# TODO: Rank karyawan berdasarkan latest_pay_rate dalam tiap department
win_pay_rank = Window.partitionBy('department_name').orderBy(F.desc('latest_pay_rate'))

df_ranked = df_transformed \
    .withColumn('pay_rank_in_dept', F.rank().over(win_pay_rank)) \
    .withColumn('tenure_rank_in_dept',
        F.rank().over(Window.partitionBy('department_name').orderBy(F.desc('tenure_years'))))

print('=== Top 5 Karyawan per Department berdasarkan Pay Rate ===')
df_ranked.filter(F.col('pay_rank_in_dept') <= 5) \
         .orderBy('department_name', 'pay_rank_in_dept') \
         .select('department_name', 'full_name', 'job_title', 'latest_pay_rate',
                 'tenure_years', 'pay_rank_in_dept') \
         .show(20, truncate=False)

=== Top 5 Karyawan per Department berdasarkan Pay Rate ===
+---------------+-------------------+-----------------------------+---------------+------------+----------------+
|department_name|full_name          |job_title                    |latest_pay_rate|tenure_years|pay_rank_in_dept|
+---------------+-------------------+-----------------------------+---------------+------------+----------------+
|Engineering    |Terri Duffy        |Vice President of Engineering|63.46          |18.3        |1               |
|Engineering    |Roberto Tamburello |Engineering Manager          |43.27          |18.52       |2               |
|Engineering    |Michael Sullivan   |Senior Design Engineer       |36.06          |15.38       |3               |
|Engineering    |Gail Erickson      |Design Engineer              |32.69          |18.36       |4               |
|Engineering    |Jossef Goldberg    |Design Engineer              |32.69          |18.31       |4               |
|Engineering    |Sharon Salav

In [41]:
# TODO: Pay bracket analysis — groupBy pay range untuk lihat pola tenure vs pay
df_pay_bracket = df_transformed \
    .withColumn('pay_bracket',
        F.when(F.col('latest_pay_rate') < 15, 'Low (<15)')
         .when(F.col('latest_pay_rate') < 25, 'Mid (15-25)')
         .when(F.col('latest_pay_rate') < 40, 'High (25-40)')
         .otherwise('Top (40+)')
    )

print('=== Rata-rata Tenure per Pay Bracket ===')
df_pay_bracket.groupBy('pay_bracket') \
    .agg(
        F.count('employee_id').alias('jumlah_karyawan'),
        F.round(F.avg('tenure_years'), 2).alias('avg_tenure_years'),
        F.round(F.avg('latest_pay_rate'), 2).alias('avg_pay_rate')
    ) \
    .orderBy('avg_pay_rate') \
    .show()

=== Rata-rata Tenure per Pay Bracket ===
+------------+---------------+----------------+------------+
| pay_bracket|jumlah_karyawan|avg_tenure_years|avg_pay_rate|
+------------+---------------+----------------+------------+
|   Low (<15)|            131|           17.21|       11.71|
| Mid (15-25)|             26|           17.06|       15.00|
|High (25-40)|             25|           17.39|       26.37|
|   Top (40+)|              3|           18.04|       63.62|
+------------+---------------+----------------+------------+



## 6. Load — Simpan ke Hive Curated

In [42]:
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# Tambahkan load_timestamp
df_final = df_transformed.withColumn('load_timestamp', F.current_timestamp())

# TODO: Simpan ke adventureworks_curated.fact_hr_workforce
# mode: overwrite, saveAsTable
df_final.write \
    .mode('overwrite') \
    .saveAsTable('adventureworks_curated.fact_hr_workforce')

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_hr_workforce')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(5, truncate=False)

=== Verifikasi ===
Total rows: 185
+-----------+------------------+-----------------------------+---------------+----------+------------+---------------+--------+-----------------+--------------------------+
|employee_id|full_name         |job_title                    |department_name|hire_date |tenure_years|latest_pay_rate|is_mover|dept_change_count|load_timestamp            |
+-----------+------------------+-----------------------------+---------------+----------+------------+---------------+--------+-----------------+--------------------------+
|2          |Terri Duffy       |Vice President of Engineering|Engineering    |2008-01-31|18.3        |63.46          |false   |0                |2026-05-17 05:50:50.525692|
|3          |Roberto Tamburello|Engineering Manager          |Engineering    |2007-11-11|18.52       |43.27          |false   |0                |2026-05-17 05:50:50.525692|
|5          |Gail Erickson     |Design Engineer              |Engineering    |2008-01-06|18.36      

## 7. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [43]:
# ── Query 1 (WAJIB): Rata-rata TenureYears per Department ─────────────────
# Jawab Business Question 1: department mana yang tenure rata-ratanya lebih tinggi?

print('=== Query 1: Rata-rata TenureYears per Department ===')
spark.sql("""
    SELECT
        department_name                 AS DepartmentName,
        COUNT(employee_id)              AS TotalKaryawan,
        ROUND(AVG(tenure_years), 2)     AS AvgTenureYears,
        ROUND(MIN(tenure_years), 2)     AS MinTenureYears,
        ROUND(MAX(tenure_years), 2)     AS MaxTenureYears
    FROM adventureworks_curated.fact_hr_workforce
    GROUP BY department_name
    ORDER BY AvgTenureYears DESC
""").show()

=== Query 1: Rata-rata TenureYears per Department ===
+--------------+-------------+--------------+--------------+--------------+
|DepartmentName|TotalKaryawan|AvgTenureYears|MinTenureYears|MaxTenureYears|
+--------------+-------------+--------------+--------------+--------------+
|   Engineering|            6|         17.37|         15.33|         18.52|
|    Production|          179|         17.22|         16.19|         19.88|
+--------------+-------------+--------------+--------------+--------------+



In [44]:
# ── Query 2 (WAJIB): Top 5 Karyawan Pay Rate Tertinggi + Tenure ───────────
# Jawab Business Question 2: korelasi pay rate vs tenure

print('=== Query 2: Top 5 Karyawan Pay Rate Tertinggi ===')
spark.sql("""
    SELECT
        full_name                                                       AS FullName,
        job_title                                                       AS JobTitle,
        department_name                                                 AS DepartmentName,
        latest_pay_rate                                                 AS LatestPayRate,
        tenure_years                                                    AS TenureYears,
        RANK() OVER (PARTITION BY department_name ORDER BY latest_pay_rate DESC) AS RankInDept
    FROM adventureworks_curated.fact_hr_workforce
    ORDER BY latest_pay_rate DESC
    LIMIT 5
""").show(truncate=False)

=== Query 2: Top 5 Karyawan Pay Rate Tertinggi ===
+------------------+-----------------------------+--------------+-------------+-----------+----------+
|FullName          |JobTitle                     |DepartmentName|LatestPayRate|TenureYears|RankInDept|
+------------------+-----------------------------+--------------+-------------+-----------+----------+
|James Hamilton    |Vice President of Production |Production    |84.13        |17.29      |1         |
|Terri Duffy       |Vice President of Engineering|Engineering   |63.46        |18.3       |1         |
|Roberto Tamburello|Engineering Manager          |Engineering   |43.27        |18.52      |2         |
|Michael Sullivan  |Senior Design Engineer       |Engineering   |36.06        |15.38      |3         |
|Gail Erickson     |Design Engineer              |Engineering   |32.69        |18.36      |4         |
+------------------+-----------------------------+--------------+-------------+-----------+----------+



In [45]:
# ── Query 3 (WAJIB): Persentase IsMover = TRUE ────────────────────────────
# Jawab Business Question 3: berapa % karyawan yang pernah pindah dept?

print('=== Query 3: Persentase IsMover per Department ===')
spark.sql("""
    SELECT
        department_name                                                         AS DepartmentName,
        COUNT(employee_id)                                                      AS TotalKaryawan,
        SUM(CAST(is_mover AS INT))                                              AS TotalMover,
        ROUND(SUM(CAST(is_mover AS INT)) / COUNT(employee_id) * 100, 2)         AS MoverPct
    FROM adventureworks_curated.fact_hr_workforce
    GROUP BY department_name
    ORDER BY department_name DESC
""").show()

=== Query 3: Persentase IsMover per Department ===
+--------------+-------------+----------+--------+
|DepartmentName|TotalKaryawan|TotalMover|MoverPct|
+--------------+-------------+----------+--------+
|    Production|          179|         0|     0.0|
|   Engineering|            6|         0|     0.0|
+--------------+-------------+----------+--------+



In [46]:
# ── Query 4 (BONUS): Korelasi Pay Rate vs Tenure — Group by Pay Bracket ───
# Apakah karyawan dengan pay rate lebih tinggi cenderung punya tenure lebih lama?

print('=== Query 4 (Bonus): Tenure per Pay Bracket ===')
spark.sql("""
    SELECT
        CASE
            WHEN latest_pay_rate < 15 THEN 'Low (<15)'
            WHEN latest_pay_rate < 25 THEN 'Mid (15-25)'
            WHEN latest_pay_rate < 40 THEN 'High (25-40)'
            ELSE 'Top (40+)'
        END                                         AS pay_bracket,
        COUNT(employee_id)                          AS jumlah_karyawan,
        ROUND(AVG(tenure_years), 2)                 AS avg_tenure_years,
        ROUND(AVG(latest_pay_rate), 2)              AS avg_pay_rate
    FROM adventureworks_curated.fact_hr_workforce
    GROUP BY 1
    ORDER BY avg_pay_rate ASC
""").show()

=== Query 4 (Bonus): Tenure per Pay Bracket ===
+------------+---------------+----------------+------------+
| pay_bracket|jumlah_karyawan|avg_tenure_years|avg_pay_rate|
+------------+---------------+----------------+------------+
|   Low (<15)|            131|           17.21|       11.71|
| Mid (15-25)|             26|           17.06|       15.00|
|High (25-40)|             25|           17.39|       26.37|
|   Top (40+)|              3|           18.04|       63.62|
+------------+---------------+----------------+------------+



In [47]:
# ── Query 5 (BONUS): Distribusi JobTitle per Department ───────────────────

print('=== Query 5 (Bonus): Top 5 JobTitle per Department ===')
# TODO: groupBy department_name, job_title → count → rank per dept → filter rank <= 5
spark.sql("""
    WITH job_counts AS (
        SELECT
            department_name,
            job_title,
            COUNT(employee_id)  AS jumlah,
            RANK() OVER (
                PARTITION BY department_name
                ORDER BY COUNT(employee_id) DESC
            )                   AS rnk
        FROM adventureworks_curated.fact_hr_workforce
        GROUP BY department_name, job_title
    )
    SELECT department_name, job_title, jumlah, rnk
    FROM job_counts
    WHERE rnk <= 5
    ORDER BY department_name, rnk
""").show(20, truncate=False)

=== Query 5 (Bonus): Top 5 JobTitle per Department ===
+---------------+-----------------------------+------+---+
|department_name|job_title                    |jumlah|rnk|
+---------------+-----------------------------+------+---+
|Engineering    |Design Engineer              |3     |1  |
|Engineering    |Vice President of Engineering|1     |2  |
|Engineering    |Engineering Manager          |1     |2  |
|Engineering    |Senior Design Engineer       |1     |2  |
|Production     |Production Technician - WC50 |26    |1  |
|Production     |Production Technician - WC60 |26    |1  |
|Production     |Production Technician - WC40 |26    |1  |
|Production     |Production Technician - WC30 |25    |4  |
|Production     |Production Technician - WC20 |22    |5  |
+---------------+-----------------------------+------+---+



In [48]:
# Cleanup
spark.stop()
print('Done! Cek tabel adventureworks_curated.fact_hr_workforce di DBeaver/HiveServer2.')

Done! Cek tabel adventureworks_curated.fact_hr_workforce di DBeaver/HiveServer2.
